# 01 - Load Controls

In [ ]:
root = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/resources/datasets/AtrophyMapping_Controls/cat12_mwp_processed'
gm = '*mwp1*resam*'
wm = '*mwp2*resam*'
csf = '*mwp3*resam*'

In [ ]:
from calvin_utils.vbm_utils.loading import import_segments
ctrl_dict = import_segments(root, gm, wm, csf, drop_substring=None)

# 02 - Load Experimentals

In [ ]:
root = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/DBS_Repositories/BWHRepository/CBCTLocalizations/BIDS/CbctDbs0468/derivatives/leaddbs/sub-CbctDbs0468/preprocessing/anat/mri'
gm = 'mwp1*resam*'
wm = 'mwp2*resam*'
csf = 'mwp3*resam*'

In [ ]:
from calvin_utils.vbm_utils.loading import import_segments
expt_dict = import_segments(root, gm, wm, csf, sub_id_index=None, sub_id_str=None, drop_substring=None)

Set Columns to Exact Subject Name
- The columns should be the exact subject name so that it would match the BIDs directory you intend to save it to. 
    - For example, column 1 would be E33508535_mwp3T1_resampled.nii, and become E33508535. This would ensure the function below:
    > save_nifti_to_bids(atrophy, bids_base_dir=base_directory, mask_path=mask_path,analysis='unthresholded_tissue_segment_z_scores', ses='ses-01', dry_run=False)
    - would save each atrophy file to: 'base_directory/sub-E33508535/ses-01/', with an example being: 'base_directory/sub-E33508535/ses-01/E33508535_composite_atrophy.nii'
- An example of your current colnames are below:

In [ ]:
first_key = next(iter(expt_dict))
print(expt_dict[first_key].columns)

Write the string pre- and post- the patient ID
- if no string prior to the subject ID, leave pre=''. Same for post. 

In [ ]:
pre = 'sub-'
post = '_ses'

Run this until you get the colnames correct. 

In [ ]:
from calvin_utils.file_utils.import_functions import GiiNiiFileImport
for k in expt_dict.keys():
    expt_dict[k] = GiiNiiFileImport.splice_colnames(expt_dict[k], pre=pre, post=post)
first_key = next(iter(expt_dict))
print(expt_dict[first_key].columns)

# Generate Z-Scored Atrophy Maps for Each Segment

In [ ]:
from calvin_utils.vbm_utils.processing import process_atrophy
atrophy, atrophy_t, stats = process_atrophy(expt_dict, ctrl_dict, threshold=0.2)

# Generate H-Scored (Composite) Atrophy Maps

In [ ]:
from calvin_utils.vbm_utils.composite_atrophy_mapper import generate_norm_map
z_ctrl, _, _ = process_atrophy(data_dict=ctrl_dict, ctrl_dict=ctrl_dict, threshold=0.2)
h, mean, std = generate_norm_map(pt_dict=atrophy, ctrl_dict=z_ctrl)
atrophy['composite'] = h
atrophy_t['composite'] = h.where(h > 0, 0)

# Save the Z-Scored Maps

Where to save?

In [ ]:
base_directory = '/Users/cu135/hires_backdrops/test/full_atrophy_only'

What session are the file associated with? This will determine where the files are saved:
    - For example dataset/sub-XX/ses-pre_operative
    - If unsure, leave as None. This will create the default ses-01 subfolder. 

In [ ]:
session = 'ses-01'

Unthresholded Maps

In [ ]:
import os
mask_path = os.path.join(os.path.dirname(os.path.abspath('')), 'rois', 'MNI152_T1_2mm_brain_mask.nii')

Save the files

In [ ]:
from calvin_utils.vbm_utils.processing import save_nifti_to_bids
save_nifti_to_bids(atrophy, bids_base_dir=base_directory, mask_path=mask_path,
                   analysis='unthresholded_tissue_segment_z_scores', ses=session, dry_run=False)

Thresholded Maps - The 'Real' Atrophy


In [ ]:
from calvin_utils.vbm_utils.processing import save_nifti_to_bids
save_nifti_to_bids(atrophy_t, bids_base_dir=base_directory, mask_path=mask_path,
                   analysis='thresholded_tissue_segment_z_scores', ses=session, dry_run=False)

All Done. Enjoy your atrophy seeds.

--Calvin